# Employee Analysis Notebook
Reads and explores data from `APP_PULSE.US.EMPLOYEE` using Snowpark.

In [ ]:
# Get active Snowflake session - no credentials needed in Snowflake Notebooks
from snowflake.snowpark.context import get_active_session
import snowflake.snowpark.functions as F

session = get_active_session()
print('Session ready:', session.get_current_database(), session.get_current_schema())

## 1. Load Employee Data

In [ ]:
df = session.table('APP_PULSE.US.EMPLOYEE')
df.show()

## 2. Row Count & Schema

In [ ]:
print('Row count:', df.count())
df.printSchema()

## 3. Salary Summary by Department

In [ ]:
df.group_by('DEPTNO').agg(
    F.count('EMPNO').alias('HEADCOUNT'),
    F.avg('SAL').alias('AVG_SAL'),
    F.max('SAL').alias('MAX_SAL'),
    F.min('SAL').alias('MIN_SAL')
).sort('DEPTNO').show()

## 4. Annual Salary using UDF

In [ ]:
df.select(
    'EMPNO',
    'ENAME',
    'JOB',
    'SAL',
    F.call_function('APP_PULSE.US.GET_ANNUAL_SALARY', F.col('SAL')).alias('ANNUAL_SAL')
).sort('ANNUAL_SAL', ascending=False).show()

## 5. Filter High Earners (SAL > 2000)

In [ ]:
df.filter(F.col('SAL') > 2000).select('EMPNO', 'ENAME', 'JOB', 'DEPTNO', 'SAL').sort('SAL', ascending=False).show()